# Week 4 Day 5: DateTime 掌握
## 時間序列基礎完整指南

**學習目標：**
- 掌握 15+ 個 to_datetime 應用
- 提取年月日、季度等時間成分
- 深入理解 Period 和 Timedelta
- 實戰時間篩選與分析

**時間估計：** 2.5 小時

In [ ]:
# ========== 環境設定 ==========
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 顯示設定
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# 時間計算
from datetime import datetime, timedelta

print("✅ 環境設定完成")

---
## Part 1: 環境設置與資料載入

In [ ]:
# 創建模擬訂單數據
np.random.seed(42)

orders = pd.DataFrame({
    'order_id': range(1001, 1051),
    'order_date': pd.date_range('2023-01-01', periods=50, freq='D'),
    'purchase_date_str': ['2023-01-01', '2023-01-02', '2023-01-03'] * 16 + ['2023-02-28'],
    'amount': np.random.randint(100, 5000, 50),
    'customer_id': np.random.randint(1, 11, 50)
})

print("=== 原始訂單數據 ===")
print(orders.head(10))
print(f"\n數據形狀：{orders.shape}")
print(f"數據類型：\n{orders.dtypes}")

---
## Part 2: DateTime 基礎（15 個範例）

### 2.1 to_datetime 轉換

In [ ]:
# 範例 1：將字符串轉換為 datetime
print("=== 範例 1：基礎字符串轉換 ===")
dates = pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03'])
print(f"轉換後類型：{dates.dtype}")
print(dates)

In [ ]:
# 範例 2：自定義日期格式
print("=== 範例 2：自定義日期格式 ===")
dates_custom = pd.to_datetime(['01/01/2023', '01/02/2023', '01/03/2023'], format='%m/%d/%Y')
print(f"自定義格式：{dates_custom}")

# 範例 3：處理多種格式
print("\n=== 範例 3：infer_datetime_format 自動推斷格式 ===")
dates_infer = pd.to_datetime(['2023-01-01', '2023/01/02', '01-03-2023'], infer_datetime_format=True)
print(f"推斷後：{dates_infer}")

In [ ]:
# 範例 4：處理時間戳
print("=== 範例 4：Unix 時間戳轉換 ===")
timestamps = [1672531200, 1672617600, 1672704000]  # Unix 時間戳
dates_unix = pd.to_datetime(timestamps, unit='s')
print(f"從 Unix 時間戳轉換：\n{dates_unix}")

# 範例 5：處理缺失值
print("\n=== 範例 5：處理缺失值 ===")
dates_missing = pd.to_datetime(['2023-01-01', None, '2023-01-03'])
print(f"包含缺失值：\n{dates_missing}")

In [ ]:
# 範例 6：Series 轉換
print("=== 範例 6：Series 轉換 ===")
orders['purchase_date'] = pd.to_datetime(orders['purchase_date_str'])
print(f"轉換後的 purchase_date 類型：{orders['purchase_date'].dtype}")
print(orders[['purchase_date_str', 'purchase_date']].head())

# 範例 7：強制類型轉換
print("\n=== 範例 7：errors 參數處理錯誤 ===")
invalid_dates = pd.to_datetime(['2023-01-01', 'invalid', '2023-01-03'], errors='coerce')
print(f"強制轉換（errors='coerce'）：\n{invalid_dates}")

### 2.2 時間成分提取

In [ ]:
# 確保有正確的 datetime 列
orders['order_date_dt'] = pd.to_datetime(orders['order_date'])

# 範例 8：提取年月日
print("=== 範例 8：提取年月日時分秒 ===")
orders['year'] = orders['order_date_dt'].dt.year
orders['month'] = orders['order_date_dt'].dt.month
orders['day'] = orders['order_date_dt'].dt.day
orders['hour'] = orders['order_date_dt'].dt.hour

print(orders[['order_date', 'year', 'month', 'day', 'hour']].head())

# 範例 9：提取星期和季度
print("\n=== 範例 9：提取星期和季度 ===")
orders['dayofweek'] = orders['order_date_dt'].dt.dayofweek  # 0=Monday
orders['quarter'] = orders['order_date_dt'].dt.quarter
orders['dayofyear'] = orders['order_date_dt'].dt.dayofyear

print(orders[['order_date', 'dayofweek', 'quarter', 'dayofyear']].head())

In [ ]:
# 範例 10：提取月份名稱
print("=== 範例 10：月份和星期名稱 ===")
orders['month_name'] = orders['order_date_dt'].dt.month_name()
orders['day_name'] = orders['order_date_dt'].dt.day_name()
orders['week_number'] = orders['order_date_dt'].dt.isocalendar().week

print(orders[['order_date', 'month_name', 'day_name', 'week_number']].head(10))

### 2.3 Period 時間周期

In [ ]:
# 範例 11：Period 基礎
print("=== 範例 11：Period 轉換 ===")
orders['period_month'] = orders['order_date_dt'].dt.to_period('M')  # M = Month
print(f"月度周期：\n{orders[['order_date', 'period_month']].head(10)}")

# 範例 12：不同 Period 頻率
print("\n=== 範例 12：不同 Period 頻率 ===")
orders['period_quarter'] = orders['order_date_dt'].dt.to_period('Q')
orders['period_week'] = orders['order_date_dt'].dt.to_period('W')

print(orders[['order_date', 'period_month', 'period_quarter', 'period_week']].head(10))

### 2.4 時間篩選

In [ ]:
# 範例 13：按月份篩選
print("=== 範例 13：按月份篩選 ===")
jan_orders = orders[orders['month'] == 1]
print(f"1月訂單數：{len(jan_orders)}")
print(jan_orders[['order_date', 'amount']].head())

# 範例 14：時間範圍篩選
print("\n=== 範例 14：時間範圍篩選 ===")
start_date = pd.to_datetime('2023-01-10')
end_date = pd.to_datetime('2023-01-20')
range_orders = orders[(orders['order_date_dt'] >= start_date) & (orders['order_date_dt'] <= end_date)]
print(f"2023-01-10 到 2023-01-20 的訂單數：{len(range_orders)}")
print(range_orders[['order_date', 'amount']].head())

In [ ]:
# 範例 15：季度篩選
print("=== 範例 15：季度篩選 ===")
q1_orders = orders[orders['quarter'] == 1]
print(f"Q1 訂單數：{len(q1_orders)}")
print(f"Q1 總金額：{q1_orders['amount'].sum()}")
print(f"Q1 平均訂單金額：{q1_orders['amount'].mean():.2f}")

---
## Part 3: Timedelta 計算（10 個範例）

時間差計算與統計

In [ ]:
# 創建配送數據
delivery = pd.DataFrame({
    'order_id': range(1001, 1021),
    'order_date': pd.date_range('2023-01-01', periods=20, freq='D'),
    'delivery_date': pd.date_range('2023-01-05', periods=20, freq='D'),
})

print("=== 原始配送數據 ===")
print(delivery.head())

In [ ]:
# 範例 1-2：基礎 Timedelta 計算
print("=== 範例 1-2：計算配送時間 ===")
delivery['delivery_time'] = delivery['delivery_date'] - delivery['order_date']
delivery['delivery_days'] = delivery['delivery_time'].dt.days
delivery['delivery_hours'] = (delivery['delivery_time'].dt.total_seconds() / 3600).astype(int)

print(delivery[['order_id', 'delivery_time', 'delivery_days', 'delivery_hours']].head(10))

# 範例 3：統計
print(f"\n平均配送時間：{delivery['delivery_days'].mean():.2f} 天")
print(f"最短配送時間：{delivery['delivery_days'].min()} 天")
print(f"最長配送時間：{delivery['delivery_days'].max()} 天")
print(f"配送時間標準差：{delivery['delivery_days'].std():.2f} 天")

In [ ]:
# 範例 4-5：Timedelta 操作
print("=== 範例 4-5：Timedelta 加減操作 ===")
delivery['expected_delivery_5'] = delivery['order_date'] + pd.Timedelta(days=5)
delivery['expected_delivery_7'] = delivery['order_date'] + pd.Timedelta(days=7)
delivery['is_delayed'] = delivery['delivery_date'] > delivery['expected_delivery_5']

print(delivery[['order_id', 'delivery_date', 'expected_delivery_5', 'is_delayed']].head())

# 延誤統計
print(f"\n延誤訂單數：{delivery['is_delayed'].sum()}")
print(f"延誤率：{delivery['is_delayed'].mean()*100:.1f}%")

In [ ]:
# 範例 6-8：Timedelta 統計分析
print("=== 範例 6-8：Timedelta 統計 ===")

# 按配送時間分類
delivery['delivery_category'] = pd.cut(delivery['delivery_days'], 
                                        bins=[0, 3, 5, 7, 100],
                                        labels=['特快(≤3天)', '快速(4-5天)', '標準(6-7天)', '延誤(>7天)'])

print("配送時間分布：")
print(delivery['delivery_category'].value_counts().sort_index())

# 平均值對比
print("\n各類別的平均配送時間：")
print(delivery.groupby('delivery_category')['delivery_days'].agg(['mean', 'count', 'std']))

In [ ]:
# 範例 9-10：實務應用
print("=== 範例 9-10：實務應用 ===")

# 警告訂單（可能延誤）
delivery['delivery_end_time'] = delivery['delivery_date'] + pd.Timedelta(hours=17)
delivery['time_remaining'] = delivery['delivery_end_time'] - pd.Timestamp.now()
delivery['is_alert'] = delivery['time_remaining'] < pd.Timedelta(hours=24)

print("訂單狀態概覽：")
print(f"已配送：{(delivery['delivery_date'] < pd.Timestamp.now()).sum()}")
print(f"待配送：{(delivery['delivery_date'] >= pd.Timestamp.now()).sum()}")
print(f"⚠️ 警告訂單：{delivery['is_alert'].sum()}")

---
## Part 4: Period 時間區間（8 個範例）

In [ ]:
# 創建銷售數據
sales = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=90, freq='D'),
    'amount': np.random.randint(1000, 5000, 90),
    'category': np.random.choice(['A', 'B', 'C'], 90)
})

print("=== 原始銷售數據 ===")
print(sales.head())

In [ ]:
# 範例 1-2：Period 分組
print("=== 範例 1-2：Period 分組聚合 ===")
sales['period_month'] = sales['date'].dt.to_period('M')
sales['period_week'] = sales['date'].dt.to_period('W')

# 按月份聚合
monthly_sales = sales.groupby('period_month')['amount'].agg(['sum', 'mean', 'count'])
monthly_sales.columns = ['月度銷售額', '平均訂單', '訂單數']
print("\n月度銷售統計：")
print(monthly_sales)

In [ ]:
# 範例 3-4：周度統計
print("=== 範例 3-4：週度銷售分析 ===")
weekly_sales = sales.groupby('period_week').agg({
    'amount': ['sum', 'mean', 'count'],
    'category': 'nunique'
})

weekly_sales.columns = ['周銷售額', '平均訂單', '訂單數', '商品類別數']
print(weekly_sales)

In [ ]:
# 範例 5-6：Period 比較
print("=== 範例 5-6：Period 同期比較 ===")

# 轉換為季度Period
sales['period_quarter'] = sales['date'].dt.to_period('Q')

# 季度銷售
quarterly_sales = sales.groupby('period_quarter')['amount'].sum()
print("季度銷售額：")
print(quarterly_sales)

# 同比增長
print("\n季度銷售變化：")
print(quarterly_sales.pct_change() * 100)

In [ ]:
# 範例 7-8：Period 範圍
print("=== 範例 7-8：Period 範圍操作 ===")

# 獲取特定月份的Period
target_period = pd.Period('2023-01', freq='M')
jan_data = sales[sales['period_month'] == target_period]

print(f"2023年1月數據：")
print(f"訂單數：{len(jan_data)}")
print(f"總銷售額：{jan_data['amount'].sum()}")
print(f"平均訂單額：{jan_data['amount'].mean():.2f}")

# Period 範圍
period_range = pd.period_range(start='2023-01', end='2023-03', freq='M')
print(f"\nQ1 Period 範圍：{list(period_range)}")

---
## Part 5: 實戰案例（6 個）

### 案例 1：訂單時間分析

In [ ]:
# 構建完整的訂單分析
print("=== 案例 1：訂單時間分析 ===")

# 創建詳細訂單數據
detail_orders = pd.DataFrame({
    'order_id': range(1001, 1101),
    'order_date': pd.date_range('2023-01-01', periods=100, freq='H'),
    'amount': np.random.randint(50, 1000, 100)
})

detail_orders['hour'] = detail_orders['order_date'].dt.hour
detail_orders['day_name'] = detail_orders['order_date'].dt.day_name()
detail_orders['date'] = detail_orders['order_date'].dt.date

# 高峰期分析
peak_hours = detail_orders.groupby('hour')['amount'].agg(['count', 'sum'])
peak_hours.columns = ['訂單數', '銷售額']

print("\n按小時的高峰期分析：")
print(peak_hours.sort_values('訂單數', ascending=False).head(10))

# 工作日 vs 非工作日
detail_orders['is_weekday'] = detail_orders['order_date'].dt.dayofweek < 5
weekday_analysis = detail_orders.groupby('is_weekday')['amount'].agg(['count', 'mean'])
weekday_analysis.index = ['非工作日', '工作日']
weekday_analysis.columns = ['訂單數', '平均訂單額']
print("\n工作日 vs 非工作日：")
print(weekday_analysis)

### 案例 2：月度趨勢分析

In [ ]:
# 月度趨勢
print("=== 案例 2：月度趨勢分析 ===")

sales_monthly = sales.copy()
sales_monthly['month'] = sales_monthly['date'].dt.to_period('M')

monthly_trend = sales_monthly.groupby('month').agg({
    'amount': ['sum', 'mean', 'count', 'std']
})

monthly_trend.columns = ['月銷售額', '平均訂單', '訂單數', '標準差']
monthly_trend['日均銷售額'] = monthly_trend['月銷售額'] / 31  # 簡化計算

print(monthly_trend)

# 環比增長
monthly_trend['環比增長%'] = monthly_trend['月銷售額'].pct_change() * 100
print("\n月度環比增長：")
print(monthly_trend[['月銷售額', '環比增長%']])

### 案例 3：交付時間分析

In [ ]:
# 交付時間分析
print("=== 案例 3：交付時間分析 ===")

delivery_analysis = delivery.copy()
delivery_analysis['delivery_time'] = delivery_analysis['delivery_date'] - delivery_analysis['order_date']
delivery_analysis['delivery_days'] = delivery_analysis['delivery_time'].dt.days

# 按訂單月份統計交付
delivery_analysis['order_month'] = delivery_analysis['order_date'].dt.to_period('M')

delivery_stats = delivery_analysis.groupby('order_month')['delivery_days'].agg([
    ('平均交付時間', 'mean'),
    ('最快交付', 'min'),
    ('最慢交付', 'max'),
    ('標準差', 'std'),
    ('訂單數', 'count')
])

print(delivery_stats)
print(f"\n全體平均交付時間：{delivery_analysis['delivery_days'].mean():.2f} 天")

### 案例 4：客戶活躍度分析

In [ ]:
# 客戶活躍度
print("=== 案例 4：客戶活躍度分析 ===")

# 使用訂單數據
customer_activity = orders[['customer_id', 'order_date_dt', 'amount']].copy()
customer_activity['year_month'] = customer_activity['order_date_dt'].dt.to_period('M')

# 按客戶和月份統計
activity_matrix = customer_activity.groupby(['customer_id', 'year_month']).agg({
    'amount': ['count', 'sum']
}).reset_index()

activity_matrix.columns = ['customer_id', 'month', 'order_count', 'total_amount']

print("\n客戶月度活躍度（前 10 筆）：")
print(activity_matrix.head(10))

# 客戶活躍月數
customer_months = activity_matrix.groupby('customer_id')['month'].count()
print("\n客戶活躍月數統計：")
print(f"最活躍客戶的活躍月數：{customer_months.max()}")
print(f"平均活躍月數：{customer_months.mean():.2f}")

### 案例 5：時間序列預測準備

In [ ]:
# 為時間序列預測準備數據
print("=== 案例 5：時間序列預測準備 ===")

# 按天的銷售額
daily_sales = sales.groupby('date')['amount'].sum().reset_index()
daily_sales.columns = ['date', 'sales']
daily_sales = daily_sales.sort_values('date')

# 添加時間特徵
daily_sales['day_of_week'] = pd.to_datetime(daily_sales['date']).dt.dayofweek
daily_sales['day_of_month'] = pd.to_datetime(daily_sales['date']).dt.day
daily_sales['month'] = pd.to_datetime(daily_sales['date']).dt.month
daily_sales['week_of_year'] = pd.to_datetime(daily_sales['date']).dt.isocalendar().week

# 滯後特徵
daily_sales['lag_1_day'] = daily_sales['sales'].shift(1)
daily_sales['lag_7_day'] = daily_sales['sales'].shift(7)

print("\n預測準備的時間序列（前 10 筆）：")
print(daily_sales.head(10))

print(f"\n特徵準備完成：")
print(f"時間範圍：{daily_sales['date'].min()} 到 {daily_sales['date'].max()}")
print(f"總天數：{len(daily_sales)}")
print(f"特徵列：{daily_sales.columns.tolist()}")

### 案例 6：複合時間分析

---
## 本堂課重點總結

### 核心概念
1. **to_datetime**: 靈活的日期字符串轉換
2. **時間成分提取**: .dt 訪問器的15+ 種用法
3. **Period**: 固定時間周期的聚合和比較
4. **Timedelta**: 時間差計算和統計分析
5. **時間篩選**: 多種時間範圍查詢方式

### 最佳實踐
- 總是使用 `pd.to_datetime()` 確保日期格式正確
- 為不同分析需求提取相應的時間成分
- 使用 Period 進行固定周期的聚合
- Timedelta 用於計算時間差和統計
- 結合多個時間維度進行複合分析

### 常見應用場景
- 訂單分析：時間分布、高峰期識別
- 配送分析：交付時間計算、延誤檢測
- 銷售趨勢：月度/季度/年度對比
- 客戶分析：活躍度、留存期追蹤
- 時間序列預測：特徵工程準備

### 下堂課預告
**Day 6: Resample & Rolling** - 時間序列重採樣和移動統計